# Values and the Bellman equation
**Machine Learning · Week 7 (Reinforcement learning) · Day 3**

Lesson page: https://sara-aminpour.com/tutorials/ml/w7d3.html

The **value** of a state is the return you can expect from it. The **Bellman equation** ties every
state's value to its neighbors' values, and **value iteration** solves it, sweep after sweep, when we
know the rules of the world.

In [ ]:
import numpy as np

maze = ["S...#.", ".##.#.", "...P..", ".#.##.", ".#....", "...#.F"]
rows, cols = len(maze), len(maze[0])
moves = [(-1, 0), (0, 1), (1, 0), (0, -1)]          # up, right, down, left
arrows = "↑→↓←"

def outcomes(state, action, slip=0.0):
    """Every possible (probability, next state, reward, episode over?): the same rules as on Monday."""
    results = []
    for direction, p in [(action, 1 - slip), ((action + 1) % 4, slip / 2), ((action + 3) % 4, slip / 2)]:
        if p == 0:
            continue
        r, c = state
        nr, nc = r + moves[direction][0], c + moves[direction][1]
        if not (0 <= nr < rows and 0 <= nc < cols) or maze[nr][nc] == "#":
            nr, nc = r, c
        cell = maze[nr][nc]
        reward = 1.0 if cell == "F" else -1.0 if cell == "P" else -0.04
        results.append((p, (nr, nc), reward, cell == "F"))
    return results

states = [(r, c) for r in range(rows) for c in range(cols) if maze[r][c] not in "#F"]

def q_value(V, s, a, gamma, slip):
    """One look ahead: the expected reward plus the discounted value of where we land."""
    return sum(p * (reward + (0.0 if done else gamma * V[s2])) for p, s2, reward, done in outcomes(s, a, slip))

## Value iteration
Start with $V = 0$ everywhere and repeat $V(s) \leftarrow \max_a \sum_{s'} P(s' \mid s, a)\,[\,r + \gamma V(s')\,]$
for every state, until nothing changes.

In [ ]:
def value_iteration(gamma=0.9, slip=0.0, tol=1e-8):
    V = np.zeros((rows, cols))
    for sweep in range(1, 1000):
        new = V.copy()
        for s in states:
            new[s] = max(q_value(V, s, a, gamma, slip) for a in range(4))
        change = np.abs(new - V).max()
        V = new
        if change < tol:
            return V, sweep
    return V, sweep

def greedy_policy(V, gamma, slip):
    return {s: int(np.argmax([q_value(V, s, a, gamma, slip) for a in range(4)])) for s in states}

def show(policy):
    for r in range(rows):
        print(" ".join(arrows[policy[(r, c)]] if (r, c) in policy else ("F" if maze[r][c] == "F" else "#") for c in range(cols)))

V, sweeps = value_iteration()
print(f"settled after {sweeps} sweeps; value of the start square: {V[0, 0]:.3f}")
print(np.round(V, 2) + 0.0)                       # walls and the fish show 0
show(greedy_policy(V, 0.9, 0.0))

## A slippery floor
With a 20% chance to slip sideways, every step next to the puddle is a risk, so the values drop.
Does the best route change too?

In [ ]:
V_slip, _ = value_iteration(slip=0.2)
print(f"value of the start square: no slip {V[0, 0]:.3f}, slippery {V_slip[0, 0]:.3f}")
print("best policy on a slippery floor:")
policy_slip = greedy_policy(V_slip, 0.9, 0.2)
show(policy_slip)
changed = [s for s in states if policy_slip[s] != greedy_policy(V, 0.9, 0.0)[s]]
print("squares whose best action changed:", changed)

## Your turn: the value of the random policy
Value iteration finds the **best** values. For a given policy, the Bellman *expectation* equation
averages over the policy's actions instead of taking the max:
$V^\pi(s) = \sum_a \pi(a \mid s) \sum_{s'} P(s' \mid s, a)\,[\,r + \gamma V^\pi(s')\,]$.
Iterate it for the random policy ($\pi(a \mid s) = 1/4$, no slip, $\gamma = 0.9$). Compare $V^\pi$ of the
start square with Monday's Monte Carlo estimate (about −0.54) and with the best value above.

In [ ]:
def evaluate_random_policy(gamma=0.9, slip=0.0, sweeps=500):
    V = np.zeros((rows, cols))
    for _ in range(sweeps):
        new = V.copy()
        for s in states:
            new[s] = 0.0        # TODO: the average of q_value(V, s, a, gamma, slip) over the 4 actions
        V = new
    return V

V_random = evaluate_random_policy()
print(f"random policy, start square: {V_random[0, 0]:.3f}   best policy: {V[0, 0]:.3f}")

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
def evaluate_random_policy(gamma=0.9, slip=0.0, sweeps=500):
    V = np.zeros((rows, cols))
    for _ in range(sweeps):
        new = V.copy()
        for s in states:
            new[s] = np.mean([q_value(V, s, a, gamma, slip) for a in range(4)])
        V = new
    return V

V_random = evaluate_random_policy()
print(f"random policy, start square: {V_random[0, 0]:.3f}   best policy: {V[0, 0]:.3f}")
print("The exact value matches Monday's average of 500 random episodes (about -0.54): two ways to the same number.")